# REWIND batch results: what did the pipeline find?

Reads what `pipeline.ipynb` (or `python batch.py`) saved in `outputs/batch/` and shows it. It never re-runs the
pipeline, so it is quick and safe to re-run. Pictures are small to keep this file small; commit it **without outputs**.

1. Overview per dataset · 2. Per video · 3. Gates compared · 4. MEVA: all cameras on one clock ·
5. One video in detail (timeline, keyframes, **labelled moments that got no keyframe**) · 6. Run times

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

import visualize as viz
from video_utils import format_hms

import os
OUT_ROOT = Path(os.environ.get("REWIND_OUT_ROOT", "outputs/batch"))
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.precision", 2)


def show_fig(fig):
    display(Image(viz.fig_png_bytes(fig)))
    plt.close(fig)


def show_img(img, quality=65):
    display(Image(viz.jpeg_bytes(img, quality)))


def vdir(r):
    return OUT_ROOT / r["dataset"] / r["video_id"]


def read_csv(p):
    return pd.read_csv(p) if Path(p).exists() else None


S = pd.read_csv(OUT_ROOT / "batch_summary.csv")
sel = pd.read_csv(OUT_ROOT / "selection_used.csv", keep_default_na=False)
ok = S[S.get("status_cpu", "").astype(str) == "ok"].copy() if "status_cpu" in S else S.iloc[0:0]
ok["minutes"] = ok.duration_s / 60
print(f"{len(S)} videos in the batch, {len(ok)} with stage 1-2 results")

## 1. Overview per dataset
- **active %**: share of the video the gate kept (time-weighted). Low = lots of empty footage skipped.
- **reduction**: how many fewer frames than "1 frame per second" go to the expensive models.
- **GT items with a keyframe**: share of labelled moments (MEVA activities / UCA sentences) with at least one keyframe
  inside. A moment without one can never be found by search. Compared with **uniform sampling** of the same number of
  frames: if gating isn't better than that, it isn't helping.
- **video min per CPU min**: speed of stages 1-2 (higher = faster than real time).

In [ ]:
def overview(d):
    g = d.groupby("dataset")
    t = pd.DataFrame({
        "videos": g.size(), "hours": g.duration_s.sum() / 3600,
        "active %": g.apply(lambda x: (x.active_pct * x.duration_s).sum() / x.duration_s.sum()),
        "segments": g.segments.sum(), "keyframes": g.keyframes.sum(),
        "keyframes / min": g.apply(lambda x: x.keyframes.sum() / x.minutes.sum()),
        "reduction (x)": g.apply(lambda x: x.naive_frames.sum() / max(1, x.keyframes.sum())),
        "video min per CPU min": g.apply(lambda x: x.duration_s.sum() / max(1e-9, x.cpu_s.sum())),
    })
    w = d.dropna(subset=["gt_kf_hit"]) if "gt_kf_hit" in d else d.iloc[0:0]
    if len(w):
        gw = w.groupby("dataset")
        t["videos with GT"] = gw.size()
        t["GT items"] = gw.gt_items.sum()
        t["GT items with a keyframe %"] = gw.apply(lambda x: 100 * (x.gt_kf_hit * x.gt_items).sum() / x.gt_items.sum())
        t["uniform, same count %"] = gw.apply(lambda x: 100 * (x.gt_kf_hit_uniform * x.gt_items).sum() / x.gt_items.sum())
        t["GT events a segment covers %"] = gw.apply(lambda x: 100 * (x.gt_event_recall * x.gt_items).sum() / x.gt_items.sum())
    if "gpu_s" in d and d.gpu_s.notna().any():
        t["video min per GPU min"] = g.apply(lambda x: x.duration_s.sum() / max(1e-9, x.gpu_s.fillna(0).sum()))
        t["index records"] = g.index_records.sum()
    return t

display(overview(ok).round(2))
# sanity check: does the real video length match the length the file name / annotation promised?
chk = ok.merge(sel[["dataset", "video_id", "duration_s"]].rename(columns={"duration_s": "expected_s"}), on=["dataset", "video_id"])
bad = chk[(chk.duration_s - chk.expected_s.astype(float)).abs() > 5]
print(f"videos whose real length differs from the expected length by > 5 s: {len(bad)}")
if len(bad):
    display(bad[["dataset", "video_id", "expected_s", "duration_s"]])
err = S[S[[c for c in S.columns if c.startswith("status_")]].astype(str).apply(lambda r: r.str.startswith("error").any(), axis=1)]
if len(err):
    display(Markdown(f"**{len(err)} video(s) failed** (traceback in `error_*.json` in their folder)"))
    display(err[["dataset", "video_id"] + [c for c in err.columns if c.startswith("status_")]])

## 2. Per video
Left: how much of each video the gate kept. Right: labelled moments with a keyframe, gated keyframes vs the same
number of frames spread evenly (points above the diagonal = gating helps).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})
d = ok.sort_values(["dataset", "active_pct"])
colors = d.dataset.map({"meva": "C0", "uca": "C1"}).fillna("C2")
axes[0].bar(range(len(d)), d.active_pct, color=colors)
axes[0].set(ylabel="% of video active", xticks=[], title="active share per video (blue MEVA, orange UCA)")
axes[0].grid(alpha=.25, axis="y")
if "gt_kf_hit" in d:
    w = d.dropna(subset=["gt_kf_hit"])
    for ds_, c in (("meva", "C0"), ("uca", "C1")):
        x = w[w.dataset == ds_]
        axes[1].scatter(100 * x.gt_kf_hit_uniform, 100 * x.gt_kf_hit, s=18 + x.gt_items.clip(upper=60), color=c, alpha=.7, label=ds_)
    axes[1].plot([0, 100], [0, 100], color="#888", ls=":", lw=1)
    axes[1].set(xlabel="uniform sampling, same count (%)", ylabel="gated keyframes (%)", xlim=(-3, 103), ylim=(-3, 103),
                title="GT items with a keyframe")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=.25)
fig.tight_layout()
show_fig(fig)
cols = ["dataset", "video_id", "category", "minutes", "active_pct", "segments", "keyframes", "reduction_x",
        "gt_items", "gt_kf_hit", "gt_kf_hit_uniform", "gt_event_recall"]
display(ok[[c for c in cols if c in ok]].sort_values(["dataset", "video_id"]).reset_index(drop=True))

### UCA by category
UCF-Crime videos are partly not CCTV (news / YouTube compilations with cuts, replays, several angles). A cut looks like
the whole frame moving, so such videos come out mostly "active". This table shows which categories suffer.

In [ ]:
u = ok[ok.dataset == "uca"]
if len(u):
    display(u.groupby("category").agg(videos=("video_id", "size"), minutes=("minutes", "sum"),
                                      active_pct=("active_pct", "mean"), keyframes_per_min=("keyframes", "sum"),
                                      gt_kf_hit=("gt_kf_hit", "mean"), gt_kf_hit_uniform=("gt_kf_hit_uniform", "mean"))
            .assign(keyframes_per_min=lambda t: t.keyframes_per_min / t.minutes).round(2))
else:
    print("no UCA videos")

## 3. Gates compared
The batch computes segments and keyframes for every gate (only the chosen one, `gate` in the config, saves JPGs and
feeds stages 3-4). Ground-truth numbers per gate are in each video's `gt_eval_gates.csv` / `gt_eval_keyframes.csv`.

In [ ]:
gates = [c[len("active_pct_"):] for c in ok.columns if c.startswith("active_pct_")]
rows = []
for _, r in ok.iterrows():
    gk = read_csv(vdir(r) / "gt_eval_keyframes.csv")
    gg = read_csv(vdir(r) / "gt_eval_gates.csv")
    for m in gates:
        row = {"dataset": r.dataset, "gate": m, "minutes": r.minutes, "active_pct": r[f"active_pct_{m}"],
               "keyframes": r[f"keyframes_{m}"], "gt_items": r.get("gt_items", 0) or 0}
        if gk is not None:
            gk_ = gk.set_index(gk.columns[0])
            row["kf_hit"] = gk_.loc[f"keyframes_{m}", "gt_events_with_keyframe"] if f"keyframes_{m}" in gk_.index else np.nan
        if gg is not None:
            gg_ = gg.set_index(gg.columns[0])
            row["event_recall"] = gg_.loc[m, "event_recall"] if m in gg_.index else np.nan
        rows.append(row)
G = pd.DataFrame(rows)
if len(G):
    def wavg(x, col, w):
        x = x.dropna(subset=[col]) if col in x else x.iloc[0:0]
        return (x[col] * x[w]).sum() / x[w].sum() if len(x) and x[w].sum() else np.nan
    t = G.groupby(["dataset", "gate"]).apply(lambda x: pd.Series({
        "active %": wavg(x, "active_pct", "minutes"), "keyframes": x.keyframes.sum(),
        "GT items with a keyframe %": 100 * wavg(x, "kf_hit", "gt_items"),
        "GT events a segment covers %": 100 * wavg(x, "event_recall", "gt_items")}))
    display(t.round(1))

## 4. MEVA: all cameras on one clock
Each row is a camera; x = real time of day. Orange = motion segments, black ticks = keyframes, green bars = labelled
activities (only for clips that have annotations). Same moment, several angles: this is the multi-camera view the
final system needs.

In [ ]:
m = ok[ok.dataset == "meva"].copy()
if len(m) and m.start_wallclock.astype(str).str.len().gt(0).any():
    m["t0"] = pd.to_datetime(m.start_wallclock)
    cams = sorted(m.camera_id.astype(str).unique())
    fig, ax = plt.subplots(figsize=(15, 0.8 * len(cams) + 1.2))
    for _, r in m.iterrows():
        y = cams.index(str(r.camera_id))
        t0 = r.t0
        def tt(sec):
            return matplotlib.dates.date2num(t0 + pd.to_timedelta(sec, unit="s"))
        ax.barh(y, tt(r.duration_s) - tt(0), left=tt(0), height=0.7, color="#eee", edgecolor="#ccc", lw=0.5)
        segs = read_csv(vdir(r) / "segments.csv")
        if segs is not None:
            for s in segs.itertuples():
                ax.barh(y, tt(s.end_s) - tt(s.start_s), left=tt(s.start_s), height=0.5, color="#fdae6b")
        kf = read_csv(vdir(r) / "keyframes.csv")
        if kf is not None and len(kf):
            ax.vlines([tt(x) for x in kf.timestamp_s], y - 0.3, y + 0.3, color="k", lw=0.5)
        gt = read_csv(vdir(r) / "gt.csv")
        if gt is not None:
            for g in gt.itertuples():
                ax.plot([tt(g.start_s), tt(g.end_s)], [y + 0.38, y + 0.38], color="#2ca02c", lw=3, solid_capstyle="butt")
    ax.set_yticks(range(len(cams)), cams)
    ax.xaxis_date(); ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%H:%M"))
    ax.set_xlabel("time of day"); ax.grid(alpha=.25, axis="x")
    ax.set_title("MEVA cameras: segments (orange), keyframes (black), labelled activities (green)", loc="left", fontsize=10)
    fig.tight_layout()
    show_fig(fig)
    # which activities are labelled, and how many got a keyframe
    acts = []
    for _, r in m.iterrows():
        gt = read_csv(vdir(r) / "gt.csv")
        if gt is not None and len(gt):
            acts.append(gt.assign(camera=r.camera_id))
    if acts:
        A = pd.concat(acts)
        display(A.groupby("label").agg(n=("label", "size"), with_keyframe=("has_keyframe", "mean"),
                                       mean_len_s=("end_s", lambda e: (e - A.loc[e.index, "start_s"]).mean()))
                .sort_values("n", ascending=False).round(2))
    else:
        print("none of the selected MEVA clips has an activity annotation (see pipeline.ipynb, section 1)")
else:
    print("no MEVA videos with a wall-clock start")

## 5. One video in detail
Set `VIDEO_ID` (a `video_id` from the tables above). `None` = the MEVA clip with the most labelled activities,
else the first video.

In [ ]:
VIDEO_ID = None
THUMB_W = 150

if VIDEO_ID is None:
    c = ok.sort_values("gt_items", ascending=False) if "gt_items" in ok else ok
    VIDEO_ID = (c[c.dataset == "meva"] if (c.dataset == "meva").any() else c).video_id.iloc[0]
R = ok[ok.video_id == VIDEO_ID].iloc[0]
D_ = vdir(R)
VIDEO = Path(R.path)
scores = pd.read_csv(D_ / "motion_scores.csv")
segments = pd.read_csv(D_ / "segments.csv")
kf = pd.read_csv(D_ / "keyframes.csv")
gt = read_csv(D_ / "gt.csv")
display(Markdown(f"**{R.dataset} / {VIDEO_ID}**: {R.minutes:.1f} min, {R.resolution}, {R.fps:.1f} fps, "
                 f"gate {R.gate}: {int(R.segments)} segments, {R.active_pct:.1f}% active, {int(R.keyframes)} keyframes "
                 f"({R.reduction_x:.0f}x fewer than 1 fps)" +
                 (f", {int(R.gt_items)} labelled items, {100 * R.gt_kf_hit:.0f}% with a keyframe" if gt is not None else "")))
show_img(viz.contact_sheet(VIDEO, R.fps, np.linspace(0, max(0, R.duration_s - 1), 12), cols=6, width=200))

In [ ]:
col = f"score_{R.gate}"
scale = 1 if R.gate == "fused" else 100
fig, ax = plt.subplots(figsize=(15, 3.6))
viz.plot_timeline(ax, scores, col, segments, R.threshold_start, R.threshold_stop,
                  max(1, round(1.0 * 5)), kf, f"{R.gate}: segments (shaded), keyframes (ticks)", scale=scale,
                  ylabel="ratio to threshold" if scale == 1 else "moving (%)")
if gt is not None:
    top = ax.get_ylim()[1]
    for g in gt.itertuples():
        ax.plot([g.start_s / 60, g.end_s / 60], [0.93 * top] * 2, lw=4, solid_capstyle="butt",
                color="#2ca02c" if g.has_keyframe else "#d62728")
    ax.set_title(ax.get_title(loc="left") + " | labelled items: green = has a keyframe, red = missed", loc="left", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1), fontsize=7)
fig.tight_layout()
show_fig(fig)
display(segments[["segment_id", "start_hms", "end_hms", "duration_s", "peak_score"]].head(30))

### What triggered the strongest segments
6 frames spread over each segment, and the peak frame with the MOG2 mask on top (**red** = moving pixels that count,
**yellow** = blobs the lighting filter removed). The mask is recomputed from the previous 60 s, so it is close to,
not identical with, what the full pass saw.

In [ ]:
TOP_N = 3
cfg_path = OUT_ROOT / "batch_config.json"
scan_params = json.loads(cfg_path.read_text())["scan"] if cfg_path.exists() else None
if scan_params is None:
    import motion_gating as mg
    scan_params = mg.SCAN_PARAMS
import motion_gating as mg
for s in segments.nlargest(TOP_N, "peak_score").itertuples():
    idx = [int(round(t * R.fps)) for t in np.linspace(s.start_s, max(s.start_s, s.end_s - 0.5), 6)]
    fr = viz.frames_at(VIDEO, idx)
    strip = viz.grid([viz.label(viz.thumb(fr[i], THUMB_W), format_hms(i / R.fps)[3:10], scale=0.35) for i in idx if i in fr],
                     cols=3, pad=2)
    peak, mask, removed = mg.foreground_mask_at(VIDEO, int(s.peak_frame_idx), scan_params, "mog2")
    ov = viz.label(viz.thumb(viz.overlay_mask(peak, mask, removed=removed), 2 * THUMB_W + 2),
                   f"peak {format_hms(s.peak_time_s)[3:10]}", scale=0.35)
    row = viz.grid([strip, ov], cols=2, pad=6)
    show_img(np.vstack([viz.header(f"segment {s.segment_id}: {s.start_hms[:8]}-{s.end_hms[:8]} ({s.duration_s:.0f} s)",
                                   row.shape[1]), row]))

### All keyframes
Timestamp and segment on top, reason at the bottom (blue first · red peak · green change · purple maxgap · grey heartbeat).

In [ ]:
tiles = []
for r in kf.itertuples():
    img = cv2.imread(str(D_ / "keyframes" / r.filename))
    if img is None:
        continue
    c = viz.hex_to_bgr(viz.REASON_COLORS.get(r.reason, "#000000"))
    t = viz.label(viz.thumb(img, THUMB_W), f"{r.timestamp_hms[3:10]} " + ("idle" if r.segment_id < 0 else f"s{r.segment_id}"), scale=0.33)
    tiles.append(viz.border(viz.label(t, r.reason, pos="bottom", scale=0.33, bg=c), c, 2))
show_img(viz.grid(tiles[:96], cols=8, pad=3), quality=55) if tiles else print("no keyframe JPGs")
if len(tiles) > 96:
    print(f"showing 96 of {len(tiles)}")

### Labelled moments that got no keyframe
These are the failures that matter: the search can never return them. One frame from the middle of each, with its
label. Use them as failure examples in the report (and check whether the label is even visible).

In [ ]:
if gt is None:
    print("this video has no ground truth")
else:
    gk = read_csv(D_ / "gt_eval_keyframes.csv")
    gg = read_csv(D_ / "gt_eval_gates.csv")
    if gg is not None:
        display(Markdown("**Per gate** (UCA: frame precision/recall mean little, its sentences cover the whole video)"))
        display(gg.set_index(gg.columns[0]).round(3))
    if gk is not None:
        display(Markdown("**Labelled items with at least one keyframe, per frame set**"))
        display(gk.set_index(gk.columns[0]).round(3))
    miss = gt[~gt.has_keyframe.astype(bool)]
    print(f"{len(miss)} of {len(gt)} labelled items have no keyframe")
    if len(miss):
        mid = {i: int(round((g.start_s + g.end_s) / 2 * R.fps)) for i, g in miss.head(12).iterrows()}
        fr = viz.frames_at(VIDEO, mid.values())
        tiles = [viz.label(viz.label(viz.thumb(fr[f], 240), f"{format_hms(miss.at[i, 'start_s'])[3:8]}-"
                                     f"{format_hms(miss.at[i, 'end_s'])[3:8]}", scale=0.35),
                           str(miss.at[i, "label"])[:40], pos="bottom", scale=0.32, bg=(40, 40, 160))
                 for i, f in mid.items() if f in fr]
        if tiles:
            show_img(viz.grid(tiles, cols=4))
        display(miss[["start_s", "end_s", "label"]].head(20))

### Stages 3-4 (only if the GPU phase ran)

In [ ]:
ev_dir = D_ / "events"
if (ev_dir / "sampling_comparison.csv").exists():
    display(Markdown("**Samplers compared** (which frames go to the captioner / index)"))
    display(pd.read_csv(ev_dir / "sampling_comparison.csv", index_col=0).round(3))
    if (ev_dir / "gt_eval_sampling.csv").exists():
        display(Markdown("**Labelled items with a keyframe, per sampler**"))
        display(pd.read_csv(ev_dir / "gt_eval_sampling.csv", index_col=0).round(3))
    ev = read_csv(ev_dir / "events.csv")
    if ev is not None and len(ev):
        display(Markdown(f"**Events** ({len(ev)}; most common)"))
        display(ev.key.value_counts().head(15).to_frame("count"))
    if (ev_dir / "timeline.png").exists():
        display(Image(filename=str(ev_dir / "timeline.png"), width=1000))
    recs = ev_dir / "index_records.jsonl"
    if recs.exists():
        display(Markdown("**Hand-off records** (first 5)"))
        display(pd.read_json(recs, lines=True).head())
else:
    print("no GPU results for this video (STAGES did not include 'gpu', or it failed: see error_gpu.json)")

## 6. Run times: how fast is indexing?
Supervisor's question: hours of video indexed per GPU-hour. Stages 1-2 run on the CPU, stages 3-4 on the GPU.
(Videos loaded from the stage-1 cache show a near-zero scan time; re-run with `FORCE=True` and a cleared cache for
honest numbers.)

In [ ]:
cols = [c for c in ["scan_s", "keyframes_s", "deep_gate_s", "cpu_s", "stage3_s", "stage4_s", "gpu_s"] if c in ok]
t = ok.groupby("dataset")[cols + ["duration_s"]].sum()
for c in cols:
    t[f"{c} per video-hour (min)"] = t[c] / 60 / (t.duration_s / 3600)
t["video hours"] = t.duration_s / 3600
display(t.drop(columns="duration_s").round(2).T)
if "from_cache" in ok and ok.from_cache.astype(str).eq("True").any():
    print(f"note: {int(ok.from_cache.astype(str).eq('True').sum())} video(s) had their stage-1 scan loaded from cache")